In [78]:
#!pip install torch torchvision torchinfo tqdm

In [2]:
!pip install -q comet_ml
import comet_ml
comet_ml.login(api_key='xxx')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.7 MB/s eta 0:00:00


COMET INFO: Valid Comet API Key saved in /root/.comet.config (set COMET_CONFIG to change where it is saved).


In [3]:
import os

import numpy as np
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
import torchvision.transforms as transforms
import imageio.v2 as imageio
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
class CFG:
  num_epochs = 10
  train_batch_size = 256
  test_batch_size = 512
  num_workers = 2
  lr = 3e-4
  seed = 12062026
  classes = ('Блузы и рубашки', 'Брюки', 'Верхняя одежда', 'Джинсы', 'Жакеты и пиджаки', 'Жилеты', 'Майки и футболки', 'Платья',
       'Топы с длинным рукавом', 'Трикотаж', 'Худи и толстовки', 'Шорты', 'Юбки')
  log = True
  project = 'oskelly-cv'
  workspace = 'gp5_team1'

In [6]:
df = pd.read_excel('oskelly_dataset.xlsx')

In [7]:
df['Категория'].unique()

array(['Пуховики', 'Джемперы и свитеры', 'Брюки узкие', 'Блузы',
       'Юбки макси', 'Спортивные брюки и шорты', 'Шорты', 'Футболки',
       'Худи и толстовки', 'Куртки', 'Прямые брюки', 'Лонгсливы',
       'Коктейльные платья', 'Жакеты и пиджаки', 'Прямые джинсы', 'Майки',
       'Тренчи и плащи', 'Спортивные куртки', 'Юбки-шорты',
       'Леггинсы и велосипедки', 'Зауженные джинсы', 'Рубашки',
       'Повседневные платья', 'Юбки мини', 'Кардиганы', 'Жилеты',
       'Купальники', 'Юбки миди', 'Туники', 'Пальто', 'Другие платья',
       'Вечерние платья', 'Комбинезоны', 'Бюстгалтеры', 'Шубы',
       'Костюмы с юбками', 'Сарафаны', 'Расклешенные джинсы', 'Водолазки',
       'Костюмы с брюками', 'Боди', 'Кюлоты', 'Спортивные костюмы',
       'Жилетки', 'Носки, чулки и колготы', 'Бриджи', 'Накидки и пончо',
       'Свадебные платья', 'Брюки широкие', 'Корсеты', 'Капри', 'Парки',
       'Халаты', 'Парео', 'Дубленки', 'Комплекты', 'Классические рубашки',
       'Повседневные брюки', 'Плавк

In [8]:
sl = {
    'Вечерние платья': 'Платья',
    'Другие платья': 'Платья',
    'Коктейльные платья': 'Платья',
    'Повседневные платья': 'Платья',
    'Сарафаны': 'Платья',
    'Юбки макси': 'Юбки',
    'Юбки миди': 'Юбки',
    'Юбки мини': 'Юбки',
    'Юбки-шорты': 'Юбки',
    'Брюки узкие': 'Брюки',
    'Брюки широкие': 'Брюки',
    'Прямые брюки': 'Брюки',
    'Кюлоты': 'Брюки',
    'Бриджи': 'Брюки',
    'Леггинсы и велосипедки': 'Брюки',
    'Зауженные джинсы': 'Джинсы',
    'Прямые джинсы': 'Джинсы',
    'Расклешенные джинсы': 'Джинсы',
    'Шорты': 'Шорты',
    'Блузы': 'Блузы и рубашки',
    'Рубашки': 'Блузы и рубашки',
    'Водолазки': 'Топы с длинным рукавом',
    'Лонгсливы': 'Топы с длинным рукавом',
    'Майки': 'Майки и футболки',
    'Футболки': 'Майки и футболки',
    'Туники': 'Майки и футболки',
    'Джемперы и свитеры': 'Трикотаж',
    'Кардиганы': 'Трикотаж',
    'Худи и толстовки': 'Худи и толстовки',
    'Пальто': 'Верхняя одежда',
    'Куртки': 'Верхняя одежда',
    'Пуховики': 'Верхняя одежда',
    'Парки': 'Верхняя одежда',
    'Тренчи и плащи': 'Верхняя одежда',
    'Дубленки': 'Верхняя одежда',
    'Шубы': 'Верхняя одежда',
    'Жакеты и пиджаки': 'Жакеты и пиджаки',
    'Накидки и пончо': 'Верхняя одежда',
    'Жилетки': 'Жилеты',
    'Жилеты': 'Жилеты',
    'Костюмы с брюками': 'Костюмы и комплекты',
    'Костюмы с юбками': 'Костюмы и комплекты',
    'Комплекты': 'Костюмы и комплекты',
    'Комбинезоны': 'Комбинезоны',
    'Бюстгалтеры': 'Белье',
    'Корсеты': 'Белье',
    'Боди': 'Боди',
    'Носки, чулки и колготы': 'Носки, чулки и колготы',
    'Купальники': 'Купальники и пляж',
    'Парео': 'Купальники и пляж',
    'Спортивные брюки и шорты': 'Спортивная одежда',
    'Спортивные костюмы': 'Спортивная одежда',
    'Спортивные куртки': 'Спортивная одежда',
    'Пижамы': 'Пижамы',
    'Пижамы и сорочки': 'Пижамы',
}

In [9]:
df['Категория для label'] = df['Категория'].map(sl)
df['Категория для label'].unique()

array(['Верхняя одежда', 'Трикотаж', 'Брюки', 'Блузы и рубашки', 'Юбки',
       'Спортивная одежда', 'Шорты', 'Майки и футболки',
       'Худи и толстовки', 'Топы с длинным рукавом', 'Платья',
       'Жакеты и пиджаки', 'Джинсы', 'Жилеты', 'Купальники и пляж',
       'Комбинезоны', 'Белье', 'Костюмы и комплекты', 'Боди',
       'Носки, чулки и колготы', nan], dtype=object)

In [10]:
sub = df.groupby('Категория для label')['image_path'].count().reset_index().sort_values('image_path')
top_labels = sub[sub['image_path'] > len(df)//100]['Категория для label'].values
df = df[df['Категория для label'].isin(top_labels)]
df = df.dropna(subset=['Категория для label', 'image_path']).reset_index(drop=True)

In [11]:
df = df.dropna(subset=['Категория для label']).reset_index(drop=True)
le = LabelEncoder()
df['label'] = le.fit_transform(df['Категория для label'])
le.classes_

array(['Блузы и рубашки', 'Брюки', 'Верхняя одежда', 'Джинсы',
       'Жакеты и пиджаки', 'Жилеты', 'Майки и футболки', 'Платья',
       'Топы с длинным рукавом', 'Трикотаж', 'Худи и толстовки', 'Шорты',
       'Юбки'], dtype=object)

In [12]:
df = df[['Категория для label', 'label', 'image_path']]
df

,Категория для label,label,image_path
0,Верхняя одежда,2,D:\GP5\oskelly_images\3136231.jpg
1,Трикотаж,9,D:\GP5\oskelly_images\3653158.jpg
2,Брюки,1,D:\GP5\oskelly_images\5392952.jpg
3,Блузы и рубашки,0,D:\GP5\oskelly_images\3904378.jpg
4,Юбки,12,D:\GP5\oskelly_images\4034222.jpg
...,...,...,...
7712,Майки и футболки,6,D:\GP5\oskelly_images\3031598.jpg
7713,Худи и толстовки,10,D:\GP5\oskelly_images\2924833.jpg
7714,Майки и футболки,6,D:\GP5\oskelly_images\3031206.jpeg
7715,Майки и футболки,6,D:\GP5\oskelly_images\4914091.jpg


In [13]:
df['image_path'] = df['image_path'].apply(lambda x: x.split('\\')[-1])
df

,Категория для label,label,image_path
0,Верхняя одежда,2,3136231.jpg
1,Трикотаж,9,3653158.jpg
2,Брюки,1,5392952.jpg
3,Блузы и рубашки,0,3904378.jpg
4,Юбки,12,4034222.jpg
...,...,...,...
7712,Майки и футболки,6,3031598.jpg
7713,Худи и толстовки,10,2924833.jpg
7714,Майки и футболки,6,3031206.jpeg
7715,Майки и футболки,6,4914091.jpg


In [14]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
f1 = '/content/drive/MyDrive/oskelly_images/'

In [16]:
df['image_path'] = df['image_path'].apply(lambda x: f1 + x.split('/')[-1])
df = df[df['image_path'].notna()].reset_index(drop=True)

In [17]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=CFG.seed)

In [95]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
])

#посчитаем mean и std для будущего Normilize
a, b, c = torch.zeros(3), torch.zeros(3), 0

for path in train['image_path']:
    photo = transform(imageio.imread(path))
    a += photo.sum(dim=[1, 2]) #dim отвечает за схлопывание по осям , мы оставляем только каналы, а остальное схлопнем суммой
    b += (photo ** 2).sum(dim=[1, 2])
    c += photo.shape[1] * photo.shape[2]

mean = a / c
std  = (b / c - mean ** 2) ** 0.5
mean

In [96]:
std

In [18]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize([0.7912, 0.7711, 0.7676], [0.3200, 0.3332, 0.3324])])

class OskellyDataset(Dataset):
    def __init__(self, data, transform):
        self.paths  = data['image_path'].tolist()
        self.labels = data['label'].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        img = imageio.imread(self.paths[index])
        img = self.transform(img)
        return img, self.labels[index]

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=CFG.seed)

чтобы засунуть наши картинки в DataLoader, я создала класс, который наследуюется от dataset, именно он преобразует ссылку из df в тензор через трансформер


шаблон взяла отсюда https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html


In [19]:
from comet_ml import Artifact

train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)

artifact = Artifact(name='img_section_processed', artifact_type='dataset')
artifact.add('train.csv')
artifact.add('test.csv')

exp_ds = comet_ml.Experiment(project_name=CFG.project, workspace=CFG.workspace)
exp_ds.log_artifact(artifact)
exp_ds.end()

вся документация по логам https://www.comet.com/docs/v2/guides/experiment-management/log-data/overview/$0

In [20]:
def train(model, device, train_loader, optimizer, criterion, epoch, log):
    model.train()
    train_loss = 0
    correct = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        train_loss = criterion(output, target)
        train_loss.backward()
        optimizer.step()

    tqdm.write('\nTrain set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        train_loss, 100. * correct / len(train_loader.dataset)))

    if log:
        log.log_metrics({'train_loss': float(train_loss), 'train_accuracy': correct / len(train_loader.dataset)}, epoch=epoch)

In [21]:
def test(model, device, test_loader, criterion, epoch, log):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    tqdm.write('Test set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        test_loss, 100. * correct / len(test_loader.dataset)))

    if log:
        log.log_metrics({'test_loss': float(test_loss), 'test_accuracy': correct / len(test_loader.dataset)}, epoch=epoch)

In [101]:
class Oskelly_base(torch.nn.Module):
    def __init__(self):
        super(Oskelly_base, self).__init__()

        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2)

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2)

        self.conv3 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.act3  = torch.nn.ReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2)

        self.fc1   = torch.nn.Linear(32 * 32 * 64, 256)
        self.act4  = torch.nn.Tanh()

        self.fc2   = torch.nn.Linear(256, 64)
        self.act5  = torch.nn.Tanh()

        self.fc3   = torch.nn.Linear(64, 13)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.act3(x)
        x = self.pool3(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.act4(x)
        x = self.fc2(x)
        x = self.act5(x)
        x = self.fc3(x)
        return x

In [22]:
def class2dict(f):
    return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

In [23]:
def CNN_main(model, train_df, test_df, transform, name):
    if CFG.log:
      experiment = comet_ml.Experiment(
            project_name=CFG.project,
            workspace=CFG.workspace)
      experiment.set_name(name)
      experiment.log_parameters(class2dict(CFG))

    use_cuda = torch.cuda.is_available()
    seed_everything(CFG.seed)
    device = torch.device("cuda" if use_cuda else "cpu")
    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': True} if use_cuda else {}

    train_loader = DataLoader(OskellyDataset(train_df, transform),
                              batch_size=CFG.train_batch_size, shuffle=True, **kwargs)
    test_loader  = DataLoader(OskellyDataset(test_df, transform),
                              batch_size=CFG.test_batch_size, shuffle=False, **kwargs)

    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=CFG.lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, experiment)
        test(model, device, test_loader, criterion, epoch, experiment)

    print('Training is ended!')
    preds = []
    answ = []
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data.to(device))
            preds += output.argmax(dim=1).cpu().tolist()
            answ += target.tolist()
    macro_f1 = f1_score(answ, preds, average='macro')

    if CFG.log:
        experiment.log_metric('test_macro_f1', macro_f1)
        experiment.log_confusion_matrix(y_true=answ, y_predicted=preds, labels=list(le.classes_))
        torch.save(model.state_dict(), 'cnn_base.pt')
        experiment.log_model('cnn_base', 'cnn_base.pt')
        experiment.get_artifact("img_section_processed")
        experiment.end()

In [104]:
model_CNN = Oskelly_base()

In [105]:
CNN_main(model_CNN, train_df, test_df, transform, 'task-img__arch-cnn_scratch__base')

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/dd344e8dd8914f889cf42a49636212f4




Epoch: 1


100%|██████████| 25/25 [12:13<00:00, 29.35s/it]


Train set: Average loss: 1.8270, Accuracy: 35.54%



/tmp/ipykernel_331/381285098.py:23: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  log.log_metrics({'train_loss': float(train_loss),


...

Test set: Average loss: 0.4797, Accuracy: 66.77%

Epoch: 10


100%|██████████| 25/25 [02:56<00:00,  7.05s/it]


Train set: Average loss: 0.6382, Accuracy: 79.31%


Test set: Average loss: 0.6027, Accuracy: 67.29%
Training is ended!


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : task-img__arch-cnn_scratch__base
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-cv/dd344e8dd8914f889cf42a49636212f4
COMET INFO:   Downloads:
COMET INFO:     artifacts : 1
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [25]           : (0.7279917597770691, 2.574066162109375)
COMET INFO:     test_accuracy [10]  : (0.4637305699481865, 0.6787564766839378)
COMET INFO:     test_loss [10]      : (0.4248583912849426, 1.331132173538208)
COMET INFO:     test_macro_f1       : 0.5283002879334547
COMET INFO:     train_accuracy [10] : (0.35541875911226306, 0.7931313785841568)
COMET INFO:     train_loss [10]     : (0

In [106]:
class Oskelly_bn_dout(torch.nn.Module):
    def __init__(self):
        super(Oskelly_bn_dout, self).__init__()
        self.batch_norm0 = torch.nn.BatchNorm2d(3)

        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.batch_norm1 = torch.nn.BatchNorm2d(16)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2)

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.batch_norm2 = torch.nn.BatchNorm2d(32)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2)

        self.conv3 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.batch_norm3 = torch.nn.BatchNorm2d(64)
        self.act3  = torch.nn.ReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2)

        self.fc1   = torch.nn.Linear(32 * 32 * 64, 256)
        self.batch_norm4 = torch.nn.BatchNorm1d(256)
        self.act4  = torch.nn.Tanh()
        self.dropout1 = nn.Dropout(0.3)

        self.fc2   = torch.nn.Linear(256, 64)
        self.act5  = torch.nn.Tanh()
        self.dropout2 = nn.Dropout(0.25)

        self.fc3   = torch.nn.Linear(64, 13)

    def forward(self, x):
        x = self.batch_norm0(x)
        x = self.pool1(self.act1(self.batch_norm1(self.conv1(x))))
        x = self.pool2(self.act2(self.batch_norm2(self.conv2(x))))
        x = self.pool3(self.act3(self.batch_norm3(self.conv3(x))))
        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.dropout1(self.act4(self.batch_norm4(self.fc1(x))))
        x = self.dropout2(self.act5(self.fc2(x)))
        x = self.fc3(x)
        return x

In [107]:
model_CNN = Oskelly_bn_dout()
CNN_main(model_CNN, train_df, test_df, transform, 'task-img__arch-cnn_scratch__bn_dout')

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/05e54c27510a4a8dbf920b16ed157b4c

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.



Epoch: 1


100%|██████████| 25/25 [02:57<00:00,  7.09s/it]


Train set: Average loss: 1.7821, Accuracy: 48.83%


Test set: Average loss: 1.5789, Accuracy: 39.96%

Epoch: 2


...

Test set: Average loss: 0.3416, Accuracy: 66.71%

Epoch: 10


100%|██████████| 25/25 [02:58<00:00,  7.16s/it]


Train set: Average loss: 0.1629, Accuracy: 98.67%


Test set: Average loss: 0.3182, Accuracy: 68.85%
Training is ended!


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : task-img__arch-cnn_scratch__bn_dout
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-cv/05e54c27510a4a8dbf920b16ed157b4c
COMET INFO:   Downloads:
COMET INFO:     artifacts : 1
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [25]           : (0.18506842851638794, 2.5729105472564697)
COMET INFO:     test_accuracy [10]  : (0.39961139896373055, 0.6884715025906736)
COMET INFO:     test_loss [10]      : (0.31819820404052734, 1.5788501501083374)
COMET INFO:     test_macro_f1       : 0.5869018465203616
COMET INFO:     train_accuracy [10] : (0.4882553053620606, 0.9867163453750203)
COMET INFO:     train_loss [10]  

In [109]:
class Oskelly_bn_dout2(torch.nn.Module):
    def __init__(self):
        super(Oskelly_bn_dout2, self).__init__()
        self.batch_norm0 = torch.nn.BatchNorm2d(3)

        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.batch_norm1 = torch.nn.BatchNorm2d(16)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2)

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.batch_norm2 = torch.nn.BatchNorm2d(32)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2)

        self.conv3 = torch.nn.Conv2d(32, 48, 3, padding=1)
        self.batch_norm3 = torch.nn.BatchNorm2d(48)
        self.act3  = torch.nn.ReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2)

        self.fc1   = torch.nn.Linear(32 * 32 * 48, 128)
        self.batch_norm4 = torch.nn.BatchNorm1d(128)
        self.act4  = torch.nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)

        self.fc2   = torch.nn.Linear(128, 32)
        self.act5  = torch.nn.ReLU()
        self.dropout2 = nn.Dropout(0.5)

        self.fc3   = torch.nn.Linear(32, 13)

    def forward(self, x):
        x = self.batch_norm0(x)
        x = self.pool1(self.act1(self.batch_norm1(self.conv1(x))))
        x = self.pool2(self.act2(self.batch_norm2(self.conv2(x))))
        x = self.pool3(self.act3(self.batch_norm3(self.conv3(x))))
        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.dropout1(self.act4(self.batch_norm4(self.fc1(x))))
        x = self.dropout2(self.act5(self.fc2(x)))
        x = self.fc3(x)
        return x

In [110]:
model_CNN = Oskelly_bn_dout2()
CNN_main(model_CNN, train_df, test_df, transform, 'task-img__arch-cnn_scratch__bn_dout2')

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/a9c55270b11c480ebb0ddc55e4556411

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.



Epoch: 1


100%|██████████| 25/25 [02:57<00:00,  7.09s/it]


Train set: Average loss: 2.2993, Accuracy: 18.06%


Test set: Average loss: 2.1830, Accuracy: 38.60%

Epoch: 2


...

Test set: Average loss: 0.7914, Accuracy: 65.41%

Epoch: 10


100%|██████████| 25/25 [02:56<00:00,  7.06s/it]


Train set: Average loss: 0.8091, Accuracy: 69.19%


Test set: Average loss: 0.6989, Accuracy: 69.24%
Training is ended!


COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : task-img__arch-cnn_scratch__bn_dout2
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-cv/a9c55270b11c480ebb0ddc55e4556411
COMET INFO:   Downloads:
COMET INFO:     artifacts : 1
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [25]           : (0.9754376411437988, 2.664111614227295)
COMET INFO:     test_accuracy [10]  : (0.3860103626943005, 0.6923575129533679)
COMET INFO:     test_loss [10]      : (0.6989114880561829, 2.1829833984375)
COMET INFO:     test_macro_f1       : 0.4932053491355462

In [24]:
class Oskelly_bn_dout3(torch.nn.Module):
    def __init__(self):
        super(Oskelly_bn_dout3, self).__init__()
        self.batch_norm0 = torch.nn.BatchNorm2d(3)

        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.batch_norm1 = torch.nn.BatchNorm2d(16)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2)

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.batch_norm2 = torch.nn.BatchNorm2d(32)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2)

        self.conv3 = torch.nn.Conv2d(32, 48, 3, padding=1)
        self.batch_norm3 = torch.nn.BatchNorm2d(48)
        self.act3  = torch.nn.ReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2)

        self.fc1   = torch.nn.Linear(32 * 32 * 48, 128)
        self.batch_norm4 = torch.nn.BatchNorm1d(128)
        self.act4  = torch.nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        self.fc2   = torch.nn.Linear(128, 32)
        self.act5  = torch.nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        self.fc3   = torch.nn.Linear(32, 13)

    def forward(self, x):
        x = self.batch_norm0(x)
        x = self.pool1(self.act1(self.batch_norm1(self.conv1(x))))
        x = self.pool2(self.act2(self.batch_norm2(self.conv2(x))))
        x = self.pool3(self.act3(self.batch_norm3(self.conv3(x))))
        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.dropout1(self.act4(self.batch_norm4(self.fc1(x))))
        x = self.dropout2(self.act5(self.fc2(x)))
        x = self.fc3(x)
        return x

In [25]:
model_CNN = Oskelly_bn_dout3()
CNN_main(model_CNN, train_df, test_df, transform, 'task-img__arch-cnn_scratch__bn_dout3')

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/5e67957ac7d24f91bdebbc165c86b0fe

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.



Epoch: 1


100%|██████████| 25/25 [24:49<00:00, 59.58s/it]


Train set: Average loss: 2.1808, Accuracy: 30.16%



/tmp/ipykernel_1203/381285098.py:23: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  log.log_metrics({'train_loss': float(train_loss),


...

Test set: Average loss: 0.6876, Accuracy: 67.42%

Epoch: 10


100%|██████████| 25/25 [03:09<00:00,  7.57s/it]


Train set: Average loss: 0.5097, Accuracy: 84.79%


Test set: Average loss: 0.6345, Accuracy: 69.11%
Training is ended!


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : task-img__arch-cnn_scratch__bn_dout3
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-cv/5e67957ac7d24f91bdebbc165c86b0fe
COMET INFO:   Downloads:
COMET INFO:     artifacts : 1
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [25]           : (0.594873309135437, 2.6031975746154785)
COMET INFO:     test_accuracy [10]  : (0.3471502590673575, 0.6910621761658031)
COMET INFO:     test_loss [10]      : (0.6153681874275208, 2.062790870666504)
COMET INFO:     test_macro_f1       : 0.5487083848985921
COMET INFO:     train_accuracy [10] : (0.301636157459906, 0.8478859549651709)
COMET INFO:     train_loss [10]     : 